In [5]:
import pandas as pd
import requests
import io
import sys
import os

root = "."
sys.path.append(os.path.join(root, "..", "src"))
from default import DATAPATH

## 1. Load `00_target_dictionary_synonyms.csv`

In [6]:
tds = pd.read_csv(os.path.join(DATAPATH, "chembl_processed", "00_target_dictionary_synonyms.csv"), low_memory=False)
print(f"Rows: {len(tds)}")
tds.head(3)

Rows: 17803


,tid,target_type,pref_name,tax_id,organism,chembl_id,species_group_flag,synonyms
0,1,SINGLE PROTEIN,Maltase-glucoamylase,9606.0,Homo sapiens,CHEMBL2074,0,"MGA;MGAML;MGAM;Maltase-glucoamylase;Alpha-1,4-..."
1,2,SINGLE PROTEIN,ATP-binding cassette sub-family C member 9,9606.0,Homo sapiens,CHEMBL1971,0,SUR2;ABCC9;ATP-binding cassette sub-family C m...
2,3,SINGLE PROTEIN,"cGMP-specific 3',5'-cyclic phosphodiesterase",9606.0,Homo sapiens,CHEMBL1827,0,"PDE5;PDE5A;cGMP-specific 3',5'-cyclic phosphod..."


In [7]:
tds_ids = set(tds["chembl_id"].dropna())
print(f"Unique ChEMBL IDs in target_dictionary_synonyms: {len(tds_ids)}")

Unique ChEMBL IDs in target_dictionary_synonyms: 17803


## 2. Download ChEMBL UniProt mapping from FTP

In [8]:
FTP_URL = "https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/latest/chembl_uniprot_mapping.txt"

print(f"Downloading {FTP_URL} ...")
response = requests.get(FTP_URL)
response.raise_for_status()
print(f"Downloaded {len(response.content) / 1024:.1f} KB")

Downloaded 1151.5 KB


In [9]:
# File format: ChEMBL_ID \t UniProt_ID \t Name \t Target_Type (no header)
uniprot_map = pd.read_csv(
    io.StringIO(response.text.replace("# chembl_36 target list, 28/07/2025\n", "")),
    sep="\t",
    header=None,
    names=["uniprot_id", "chembl_id", "name", "target_type"]
)
print(f"Rows: {len(uniprot_map)}")
uniprot_map.head(3)

Rows: 16675


,uniprot_id,chembl_id,name,target_type
0,P21266,CHEMBL2242,Glutathione S-transferase Mu 3,SINGLE PROTEIN
1,O00519,CHEMBL2243,Fatty-acid amide hydrolase 1,SINGLE PROTEIN
2,P19217,CHEMBL2244,Sulfotransferase 1E1,SINGLE PROTEIN


In [10]:
uniprot_ids = set(uniprot_map["chembl_id"].dropna())
print(f"Unique ChEMBL IDs in UniProt mapping: {len(uniprot_ids)}")
print(f"Target type breakdown:")
print(uniprot_map["target_type"].value_counts().to_string())

Unique ChEMBL IDs in UniProt mapping: 12904
Target type breakdown:
target_type
SINGLE PROTEIN                  10962
PROTEIN COMPLEX                  1601
PROTEIN FAMILY                   1568
PROTEIN-PROTEIN INTERACTION      1259
PROTEIN COMPLEX GROUP             478
PROTEIN NUCLEIC-ACID COMPLEX      408
SELECTIVITY GROUP                 285
CHIMERIC PROTEIN                   67
NUCLEIC-ACID                       47


## 3. Overlap

In [11]:
overlap = tds_ids & uniprot_ids
only_tds = tds_ids - uniprot_ids
only_uniprot = uniprot_ids - tds_ids

print(f"In target_dictionary_synonyms only : {len(only_tds):>6}")
print(f"In UniProt mapping only            : {len(only_uniprot):>6}")
print(f"In both                            : {len(overlap):>6}")
print()
print(f"Coverage of UniProt mapping by tds : {100 * len(overlap) / len(uniprot_ids):.1f}%")
print(f"Coverage of tds by UniProt mapping : {100 * len(overlap) / len(tds_ids):.1f}%")

In target_dictionary_synonyms only :   4899
In UniProt mapping only            :      0
In both                            :  12904

Coverage of UniProt mapping by tds : 100.0%
Coverage of tds by UniProt mapping : 72.5%


## 4. ChEMBL IDs in UniProt mapping but missing from `target_dictionary_synonyms`

In [12]:
missing = uniprot_map[uniprot_map["chembl_id"].isin(only_uniprot)].copy()
print(f"Rows in UniProt mapping not covered by target_dictionary_synonyms: {len(missing)}")
print(f"Unique ChEMBL IDs: {missing['chembl_id'].nunique()}")
print()
print("Target type breakdown among missing:")
print(missing["target_type"].value_counts().to_string())

Rows in UniProt mapping not covered by target_dictionary_synonyms: 0
Unique ChEMBL IDs: 0

Target type breakdown among missing:
Series([], )


In [13]:
missing.sort_values("chembl_id").head(20)

,uniprot_id,chembl_id,name,target_type
